# Chapter 13 &mdash; Universal Computers, and Tape Simulation by Two Stacks

**Concept 5 of the Chapter 13 decomposition:** *The Long List of Universal Computers, and Tape Simulation by Two Stacks*

Two stacks, one queue, or two counters all reach TM power &mdash; and the two-stack simulation is the clearest.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Universal-Computers-And-Two-Stacks/Concept-Universal-Computers-And-Two-Stacks.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Many restricted machines turn out to be **exactly as powerful** as a Turing machine:

* a PDA with **two stacks**,
* a machine with **one queue**,
* a machine with **two counters** (Minsky).

The two-stack simulation is the one to remember, because it explains *why* a PDA falls
short. Split the tape at the head:

* the **left stack** holds everything to the left of the head, topmost = nearest;
* the **right stack** holds the head cell and everything to its right.

**Move right** = pop the right stack, push onto the left. **Move left** = the reverse.
**Write** = replace the right stack's top.

A PDA has **one** stack, so it can see one end of its memory. Two stacks give you the
middle &mdash; and that is the whole difference between context-free and computable.

## 2. Definitions

### A two-stack tape

In [ ]:
class TwoStackTape:
    def __init__(self, tape, blank='.'):
        self.left  = []                       # reversed: top = nearest the head
        self.right = list(tape) or [blank]
        self.blank = blank
    def read(self):
        if not self.right: self.right.append(self.blank)
        return self.right[0]
    def write(self, c):
        if not self.right: self.right.append(self.blank)
        self.right[0] = c
    def move(self, d):
        if d == 'R':
            if not self.right: self.right.append(self.blank)
            self.left.append(self.right.pop(0))
        elif d == 'L':
            if not self.left: self.left.append(self.blank)
            self.right.insert(0, self.left.pop())
    def __str__(self):
        l = ''.join(self.left)
        r = ''.join(self.right) or self.blank
        return "%s[%s]%s" % (l, r[0], r[1:])

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

### Driving a Jove TM by hand on the two-stack tape

In [ ]:
def run_two_stack(T, tape, fuel=200):
    st, q, steps = TwoStackTape(tape), T["q0"], 0
    while steps < fuel:
        key = (q, st.read())
        if key not in T["Delta"]: break            # stuck = halted
        outs = sorted(T["Delta"][key])
        q2, wr, d = outs[0]
        st.write(wr); st.move(d); q = q2; steps += 1
    return q, str(st), steps

## 3. Tests

The two stacks reconstruct the tape and head position.

In [ ]:
st = TwoStackTape('0101')
print("  start        ", st)
for d in 'RRL':
    st.move(d); print("  move %s       %s" % (d, st))
assert str(TwoStackTape('0101')) == "[0]101"

A Jove TM driven by the two-stack tape gives the same answer.

In [ ]:
Flip = md2mc('''TM
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')
for t in ['0101', '111', '0']:
    q, tape, steps = run_two_stack(Flip, t)
    jove = tm_tape(Flip, t, fuel=60)[0]
    print("  %-7r two-stack %-10r (state %s)  Jove %r" % (t, tape, q, jove))
    assert tape.replace('[', '').replace(']', '').rstrip('.') == jove

Left moves work too, which is what a single stack cannot do.

In [ ]:
Back = md2mc('''TM
I : 0 ; 1 , R -> I
I : . ; . , L -> B
B : 1 ; 0 , L -> B
B : . ; . , S -> F
''')
q, tape, steps = run_two_stack(Back, '000')
print("  sweep right then back left :", tape, " state", q, " steps", steps)
assert q == 'F'

**Why one stack is not enough.**

In [ ]:
print("one stack : you can see the TOP -- one end of the memory")
print("two stacks: the head sits BETWEEN them -- you can see the middle,")
print("            and move the boundary either way")
print()
print("A PDA reads its input once, left to right, and can only consult one end")
print("of its memory.  That is exactly the gap between CFL and computable.")

The other universal models, for the record.

In [ ]:
MODELS = [("2 stacks",   "this simulation"),
          ("1 queue",    "a queue can simulate two stacks"),
          ("2 counters", "Minsky: encode two stacks as two integers"),
          ("1 counter",  "NOT universal -- strictly weaker")]
for m, why in MODELS: print("  %-12s %s" % (m, why))

## 4. Exercises


1. Simulate a two-stack machine with one queue. What is the trick?
2. Why are **two** counters enough but one not?
3. How many steps does the two-stack simulation take per TM step?

In [ ]:
# Your work for the exercises above.